# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadimran5016/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Unit of analysis + time window

**Unit of analysis:** One row = one page's daily performance record.

**Time window:** I will use the mid-panel month of **March 2026** for the verification queries and feature work.

For this lane, the page-level search and performance signals are used to identify pages that may be worth prioritizing for content review or refresh.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

**Feature fields:**  
- `ctr` — a page's click-through rate, useful as a search-performance signal.
- `avg_position` — the page's average search position, useful for understanding ranking performance.
- `impressions` — the number of search impressions, useful for understanding search visibility.
- `word_count` — the amount of content on the page, useful as a content-related signal.
- `trend_pct` — the change in performance over time, useful as a directional signal.

**Label / proxy:**  
- **Content opportunity score** — this is the proxy for this lane because the warehouse does not provide a direct label saying whether a page should be refreshed.

**Context fields:**  
- `date` — identifies the day represented by the row.
- `page` — identifies the page being analyzed.
- `query` — provides search-query context for the page's performance.

**Excluded:**  
- Any field that would only become known after a content team has already reviewed or changed a page is excluded from the feature set. This prevents information from the future decision process from leaking into the scoring features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [5]:
import duckdb

con = duckdb.connect()

print("DuckDB connection ready.")

DuckDB connection ready.


In [8]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [10]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf "
    "(TYPE huggingface, TOKEN ?)",
    [token]
)

print("Hugging Face authentication ready.")

Hugging Face authentication ready.


In [18]:
from google.colab import userdata

import duckdb

token = userdata.get("HF_TOKEN")

con.execute("DROP SECRET IF EXISTS hf")

safe_token = token.replace("'", "''")

con.execute(f"""
CREATE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{safe_token}'
)

""")

print("Hugging Face authentication configured correctly.")

Hugging Face authentication configured correctly.


In [14]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

api = HfApi(token=token)

info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset access confirmed!")
print("Dataset:", info.id)

Dataset access confirmed!
Dataset: FlyRank/internship-warehouse


In [16]:
# Check which warehouse tables are available
print(con.sql("SHOW TABLES").df())

Empty DataFrame
Columns: [name]
Index: []


In [22]:
# Check the Hugging Face secret registered in DuckDB
print(con.sql("SELECT * FROM duckdb_secrets()").df())

  name         type provider  persistent storage    scope  \
0   hf  huggingface   config       False  memory  [hf://]   

                                       secret_string  
0  name=hf;type=huggingface;provider=config;seria...  


In [23]:
print(con.sql("SELECT version()").df())

  "version"()
0      v1.3.2


In [25]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

# Remove old DuckDB Hugging Face secret
con.execute("DROP SECRET IF EXISTS hf")

# Create a fresh Hugging Face secret for hf:// access
con.execute("""
CREATE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [token])

print("Fresh Hugging Face secret created.")

Fresh Hugging Face secret created.


In [29]:
from google.colab import userdata
import requests

token = userdata.get("HF_TOKEN")

url = "https://huggingface.co/datasets/FlyRank/internship-warehouse/resolve/main/fact_content_daily_performance/month%3D2026-03/data_0.parquet"

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.get(url, headers=headers)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("File size:", len(response.content))

Status: 401
Content-Type: text/plain; charset=utf-8
File size: 138


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

**Limitation:** The current analysis is limited to the available warehouse data and the March 2026 slice. Because the warehouse access is currently returning a 401 Unauthorized error, the verification queries and real feature checks could not be completed. Therefore, the current contract should be treated as a provisional data contract until warehouse access is available and the fields and row grain can be verified on the real data.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.